# MCP Client Agent
## Plug-and-Play Tools via the Model Context Protocol

⏱ ~12 min
**Model Context Protocol (MCP)** is an open standard introduced by Anthropic in 2024 that defines
how AI agents communicate with external tool servers. Instead of hardcoding every tool inside the
agent, MCP separates concerns: the **server** owns tool logic and the **client** (the agent)
discovers and invokes tools dynamically at runtime.

By the end of this session you will understand *why* dynamic tool discovery matters, *how* the
MCP client contract works (`list_tools` then `call_tool`), and *how* to build a LangGraph agent
that speaks the protocol using an in-process mock server — no subprocess, no network required.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — What is MCP and why does it exist? |
| 2 | **The MockMCPServer** — Building a drop-in stand-in for a real server |
| 3 | **Tool Discovery** — `list_tools()` and the LLM selection step |
| 4 | **Tool Invocation** — `call_tool()` and argument marshalling |
| 5 | **Full Agent Graph** — Wiring discover → invoke → synthesize in LangGraph |
| 6 | **Multi-Server Routing** — Dispatching across two independent MCP servers |
| 7 | **Error Handling and Fallbacks** — Graceful recovery when tools fail |
| ★ | **Migration to Real MCP** (bonus) — Swapping the mock for `langchain-mcp-adapters` |

---

### Prerequisites
- Python 3.10+ with a `.venv` (requirements already installed locally)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Anthropic. (2024). *Model Context Protocol Specification.* https://spec.modelcontextprotocol.io  
> Anthropic. (2024). *MCP Introduction.* https://modelcontextprotocol.io/introduction  
> LangChain MCP Adapters: https://github.com/langchain-ai/langchain-mcp-adapters  
> LangGraph documentation: https://langchain-ai.github.io/langgraph/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon) as 'OPENAI_API_KEY'
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or invalid. "
    "Local: add to .env. Colab: add to Secrets panel."
)
print(f"API key loaded: {key[:8]}...{key[-4:]}")

---

## Part 1 — What Is MCP and Why Does It Exist?

---

### The Problem with Hardcoded Tools

The traditional approach to giving an LLM access to tools is to bind them at startup:

```python
llm.bind_tools([get_weather, calculate_tip, search_docs, ...])
```

This works for small, stable toolsets. But it breaks down as soon as:
- Tools live in separate services or processes
- The tool catalog grows beyond a few dozen functions
- Teams want to add or remove tools without redeploying the agent
- Different agents need different subsets of tools from the same pool

**MCP solves this by separating the *registry* from the *agent*.** Tools are owned by servers;
agents discover them dynamically at runtime.

---

### MCP Primitives

| Primitive | What it is | Analogy |
|-----------|------------|---------|
| **Tool** | A callable function with a name, description, and JSON Schema for args | REST POST endpoint |
| **Resource** | A read-only data source (file, URL, database row) | HTTP GET endpoint |
| **Prompt** | A reusable, parameterized system prompt template | Jinja template |
| **Server** | A process that exposes tools, resources, and prompts | Microservice |
| **Client** | The agent that connects to one or more servers | API consumer |

This workshop focuses on **Tools** — the most commonly used primitive.

---

### Transport Types

| Transport | How it works | Best for |
|-----------|--------------|----------|
| **stdio** | Agent spawns server as a subprocess; communicates over stdin/stdout | Local tools, CLI utilities |
| **HTTP + SSE** | Server is a long-running HTTP service; agent connects over the network | Remote services, multi-agent systems |
| **In-process** | Server lives in the same Python process (this workshop) | Testing, local mocks |

---

### MCP vs Function Calling vs Plugins

| Feature | Function Calling (OpenAI) | ChatGPT Plugins | MCP |
|---------|--------------------------|-----------------|-----|
| Tool definition location | Agent code | Plugin manifest | Server |
| Discovery | Hardcoded at startup | API manifest fetch | `list_tools()` at runtime |
| Multi-server | Manual | Not supported | First-class |
| Standard | Vendor-specific | Deprecated | Open spec |
| Transport | HTTP only | HTTP only | stdio, HTTP+SSE, in-process |

### MCP Architecture

```
AGENT SIDE                            SERVER SIDE
──────────────────────────────────    ──────────────────────────────────────

  User query
       │
       ▼
  ┌──────────────┐  list_tools()    ┌──────────────────────────────────────┐
  │              │ ───────────────▶ │         MCP Server                   │
  │  MCP Client  │ ◀─────────────── │                                      │
  │   (Agent)    │  [{name,desc}]   │  ┌─────────────┐  ┌───────────────┐ │
  │              │                  │  │  get_weather │  │ calculate_tip │ │
  │  LLM picks   │                  │  └─────────────┘  └───────────────┘ │
  │  tool by     │  call_tool(      │  ┌─────────────┐  ┌───────────────┐ │
  │  name from   │    name, args)   │  │  get_time   │  │  search_docs  │ │
  │  description │ ───────────────▶ │  └─────────────┘  └───────────────┘ │
  │              │ ◀─────────────── │                                      │
  │              │   result str     └──────────────────────────────────────┘
  │              │
  │  Synthesize  │
  │  final answer│
  └──────────────┘
       │
       ▼
  "The weather in Tokyo is sunny, 22 deg C."
```

**Three core ideas:**
1. Tools are **discovered at runtime**, not hardcoded in the agent
2. The **server owns** tool implementations; the client only knows the interface
3. The contract is minimal: `list_tools()` then `call_tool(name, args)` then `str`

In [ ]:
# ─── 1-A: The two-primitive contract ─────────────────────────────────────────
# MCP reduces the entire client-server interaction to two methods.
# Everything else (transport, serialisation, streaming) is infrastructure.
# Here we define the minimal interface an MCP client expects from any server.

from typing import Protocol


class MCPServerProtocol(Protocol):
    """The only two methods an MCP client needs from a server."""

    def list_tools(self) -> list[dict]:
        """Returns [{name: str, description: str}, ...]"""
        ...

    def call_tool(self, name: str, args: dict) -> str:
        """Invokes tool by name, returns result as a string."""
        ...


print("MCPServerProtocol defined.")
print("Any object with list_tools() + call_tool() satisfies this contract.")
print("This is why swapping mock for real server requires zero changes to the agent graph.")

---

## Part 2 — The MockMCPServer

---

A real MCP server runs as a separate process and communicates over stdio or HTTP. For this
workshop we build an **in-process mock** that satisfies the same two-method contract. This
lets us learn the pattern without any subprocess setup.

The mock is architecturally identical to a real server — the only difference is that tools
execute as Python callables in the same process instead of over a transport layer.

In [ ]:
# ─── 2-A: MockMCPServer implementation ───────────────────────────────────────
# Mirrors the real MCP server interface exactly.
# Registered tools have a name, a human-readable description (what the LLM reads),
# and a callable implementation (invisible to the LLM).

from dataclasses import dataclass
from typing import Callable


@dataclass
class MCPTool:
    name: str
    description: str
    fn: Callable


class MockMCPServer:
    """In-process stand-in for an MCP server — satisfies MCPServerProtocol."""

    def __init__(self):
        self._tools: dict[str, MCPTool] = {}

    def register(self, name: str, description: str, fn: Callable):
        """Register a new tool. Call this during server startup."""
        self._tools[name] = MCPTool(name=name, description=description, fn=fn)

    def list_tools(self) -> list[dict]:
        """MCP contract: returns the tool catalog — name + description only."""
        return [{"name": t.name, "description": t.description} for t in self._tools.values()]

    def call_tool(self, name: str, args: dict) -> str:
        """MCP contract: invoke tool by name, return result as string."""
        if name not in self._tools:
            return f"Error: unknown tool '{name}'"
        try:
            return str(self._tools[name].fn(**args))
        except TypeError as e:
            return f"Error: invalid arguments for '{name}': {e}"


print("MockMCPServer defined.")

In [ ]:
# ─── 2-B: Register tools and inspect the catalog ─────────────────────────────
# The LLM only ever sees the name and description.
# The implementation (lambda) is invisible to the LLM — only the server executes it.

server = MockMCPServer()

server.register(
    name="get_weather",
    description="Get current weather for a city. Args: city (str)",
    fn=lambda city: f"Sunny, 22 deg C in {city}",
)
server.register(
    name="calculate_tip",
    description="Calculate tip amount. Args: bill (float), pct (float percentage, e.g. 15)",
    fn=lambda bill, pct: f"${bill * pct / 100:.2f} tip on ${bill:.2f}",
)
server.register(
    name="get_time",
    description="Get current local time in a city. Args: city (str)",
    fn=lambda city: f"12:34 PM local time in {city}",
)
server.register(
    name="search_docs",
    description="Search internal documentation. Args: query (str)",
    fn=lambda query: f"Found 3 results for '{query}': doc_a.pdf, doc_b.md, doc_c.txt",
)

# Inspect the catalog — this is exactly what the LLM receives during discovery
catalog = server.list_tools()
print(f"Tools registered: {len(catalog)}\n")
for tool in catalog:
    print(f"  {tool['name']:20} {tool['description']}")

In [ ]:
# ─── 2-C: call_tool() walkthrough ────────────────────────────────────────────
# Manually invoke each tool to see what the agent will receive as tool_result.
# Also test the error cases — the agent must handle these gracefully.

test_calls = [
    ("get_weather",   {"city": "Tokyo"}),
    ("calculate_tip", {"bill": 47.50, "pct": 15}),
    ("get_time",      {"city": "London"}),
    ("search_docs",   {"query": "refund policy"}),
    ("unknown_tool",  {}),              # error: no such tool
    ("calculate_tip", {"bill": 30}),   # error: missing required arg 'pct'
]

print("Manual call_tool() tests:\n")
for name, args in test_calls:
    result = server.call_tool(name, args)
    status = "OK " if not result.startswith("Error") else "ERR"
    print(f"  [{status}] {name}({args})")
    print(f"         → {result}")

---

## Part 3 — Tool Discovery: list_tools() and LLM Selection

---

The **discover** step is the heart of the MCP pattern. The agent:
1. Calls `list_tools()` to get the current catalog from the server
2. Formats the catalog as text and sends it to the LLM
3. Asks the LLM: *given this query and this tool list, which tool should I call and with what arguments?*
4. Parses the LLM's JSON response to extract `tool_name` and `tool_args`

**Key insight:** The LLM is doing *tool selection*, not *tool execution*. Execution happens
separately in the `invoke` step. The LLM never sees tool implementations — only descriptions.

### Why JSON output for tool selection?

The selection response must be machine-parseable so the agent can extract the tool name and
arguments without fragile string manipulation. We prompt the LLM to respond with strict JSON
and parse it with `json.loads()`.

In [ ]:
# ─── 3-A: Imports and LLM setup ───────────────────────────────────────────────
import json
from typing import TypedDict

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# temperature=0.2 gives slight variation in tool-selection phrasing — use 0 for fully deterministic picks
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)


# Agent state — all fields the graph nodes share
# TypedDict gives LangGraph typed access to all state keys — no runtime overhead, just IDE/type-checker hints
class MCPState(TypedDict):
    query: str               # the user's original question
    available_tools: list[dict]  # catalog from list_tools()
    tool_name: str           # selected by the LLM
    tool_args: dict          # constructed by the LLM
    tool_result: str         # returned by call_tool()
    final_answer: str        # polished response for the user


print("LLM (gpt-4o-mini) and MCPState schema ready.")

In [ ]:
# ─── 3-B: The discover node ───────────────────────────────────────────────────
# This becomes Node 1 in the LangGraph graph.
# It calls list_tools(), builds the selection prompt, and returns structured output.

DISCOVER_PROMPT = """You have access to these tools from an MCP server:
{tools}

For this query: "{query}"
Which tool should be called? Respond as JSON only (no markdown, no explanation):
  {{"tool": "tool_name", "args": {{...}}}}
If no tool fits, respond:
  {{"tool": "none", "args": {{}}}}"""

# The LLM picks the tool and constructs its args from tool descriptions alone — no code introspection needed

def discover_and_select(state: MCPState) -> dict:
    """Node 1 — discover tools and ask LLM to select one."""
    # Step 1: fetch the live tool catalog
    tools = server.list_tools()
    tools_text = "\n".join(f"- {t['name']}: {t['description']}" for t in tools)

    # Step 2: ask the LLM to pick a tool and construct its arguments
    prompt = DISCOVER_PROMPT.format(tools=tools_text, query=state["query"])
    response = llm.invoke([HumanMessage(content=prompt)])

    # Step 3: parse the JSON response
    try:
        text = response.content.strip()
        if text.startswith("```"):  # strip markdown code fence if present
            text = "\n".join(text.split("\n")[1:-1])
        parsed = json.loads(text)
        print(f"  [discover] selected='{parsed.get('tool')}' args={parsed.get('args')}")
        return {
            "available_tools": tools,
            "tool_name": parsed.get("tool", "none"),
            "tool_args": parsed.get("args", {}),
        }
    except (json.JSONDecodeError, AttributeError) as e:
        print(f"  [discover] parse error: {e} — falling back to 'none'")
        return {"available_tools": tools, "tool_name": "none", "tool_args": {}}


# Smoke test discover_and_select in isolation
print("Testing discover_and_select in isolation:\n")
_blank: MCPState = {
    "query": "What is the weather in Paris?",
    "available_tools": [], "tool_name": "",
    "tool_args": {}, "tool_result": "", "final_answer": "",
}
_out = discover_and_select(_blank)
print(f"  tool_name  : {_out['tool_name']}")
print(f"  tool_args  : {_out['tool_args']}")
print(f"  catalog    : {len(_out['available_tools'])} tools available")

---

## Part 4 — Tool Invocation: call_tool() and Argument Marshalling

---

The **invoke** step receives the `tool_name` and `tool_args` chosen by the LLM and calls
`call_tool()` on the server. Two things can go wrong here:

1. **Unknown tool** — the LLM hallucinated a tool name that does not exist on the server
2. **Invalid arguments** — the LLM passed wrong types or omitted a required parameter

Both errors should be caught and returned as descriptive strings rather than exceptions,
so the synthesize step can include them in the final answer gracefully rather than crashing.

In [ ]:
# ─── 4-A: The invoke node ────────────────────────────────────────────────────

def invoke_tool(state: MCPState) -> dict:
    """Node 2 — call the selected tool on the server."""
    if state["tool_name"] == "none":
        print("  [invoke] no tool selected — skipping")
        return {"tool_result": "No matching tool found."}
    # call_tool() is the only other MCP primitive — name + args dict, returns a plain string
    result = server.call_tool(state["tool_name"], state["tool_args"])
    print(f"  [invoke] {state['tool_name']}({state['tool_args']}) → {result}")
    return {"tool_result": result}


# Test invoke_tool in isolation — pass a state dict that simulates discover output
print("Testing invoke_tool in isolation:\n")
_base: MCPState = {
    "query": "", "available_tools": [], "tool_name": "",
    "tool_args": {}, "tool_result": "", "final_answer": "",
}
test_invocations = [
    {"tool_name": "get_weather",   "tool_args": {"city": "Sydney"}},
    {"tool_name": "calculate_tip", "tool_args": {"bill": 80.0, "pct": 20}},
    {"tool_name": "none",          "tool_args": {}},
    {"tool_name": "bad_tool",      "tool_args": {}},
]
for case in test_invocations:
    out = invoke_tool({**_base, **case})
    print(f"  result: {out['tool_result']}")

In [ ]:
# ─── 4-B: The synthesize node ─────────────────────────────────────────────────
# Takes the raw tool result string and generates a natural-language final answer.
# This is where "Sunny, 22 deg C in Tokyo" becomes a polished user response.

ANSWER_PROMPT = """User query: {query}
Tool used: {tool_name}
Tool result: {tool_result}

Write a concise, helpful answer to the user's query using the tool result above.
If the tool result contains an error, acknowledge it and offer to help differently."""


def synthesize(state: MCPState) -> dict:
    """Node 3 — turn the raw tool result into a polished answer."""
    prompt = ANSWER_PROMPT.format(
        query=state["query"],
        tool_name=state["tool_name"],
        tool_result=state["tool_result"],
    )
    # Alternative: stream with llm.stream() to show tokens progressively for long synthesis responses
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [synthesize] answer ready ({len(response.content)} chars)")
    return {"final_answer": response.content}


print("All three nodes defined: discover_and_select, invoke_tool, synthesize")

---

## Part 5 — The Full Agent Graph

---

Now we wire the three nodes into a LangGraph `StateGraph`. The graph is a linear
pipeline: **discover → invoke → synthesize → END**.

```
START
  │
  ▼
┌──────────────────────────────────────────────┐
│  discover                                    │
│  list_tools() + LLM selects tool + args      │
└──────────────────────────┬───────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────┐
│  invoke                                      │
│  call_tool(name, args) → raw result string   │
└──────────────────────────┬───────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────┐
│  synthesize                                  │
│  LLM turns result into polished answer       │
└──────────────────────────┬───────────────────┘
                           │
                           ▼
                          END
```

State flows through all nodes. Each node reads what it needs from `state` and returns only
the fields it updates — LangGraph merges the returned dict back into the shared state.

In [ ]:
# ─── 5-A: Build the LangGraph workflow ────────────────────────────────────────
from langgraph.graph import END, StateGraph


# StateGraph(MCPState) binds the typed state schema — every node receives and returns a subset of MCPState
def create_workflow():
    graph = StateGraph(MCPState)

    graph.add_node("discover",   discover_and_select)
    graph.add_node("invoke",     invoke_tool)
    graph.add_node("synthesize", synthesize)

    graph.set_entry_point("discover")
    graph.add_edge("discover",   "invoke")
    graph.add_edge("invoke",     "synthesize")
    graph.add_edge("synthesize", END)

    # compile() locks the graph topology — after this, nodes and edges cannot be added or removed
    return graph.compile()


app = create_workflow()
print("Workflow compiled — nodes: discover → invoke → synthesize → END")

In [ ]:
# ─── 5-B: Run the full pipeline on sample queries ────────────────────────────

SAMPLE_QUERIES = [
    "What is the weather like in Tokyo?",
    "Calculate 15% tip on a $47.50 bill.",
    "What time is it in London right now?",
    "Search for documents about onboarding.",
    "What is the capital of France?",   # no matching tool — tests 'none' path
]


def run_query(query: str) -> dict:
    """Run a query through the compiled workflow."""
    return app.invoke(
        {
            "query": query,
            "available_tools": [],
            "tool_name": "",
            "tool_args": {},
            "tool_result": "",
            "final_answer": "",
        }
    )


print("=" * 60)
for query in SAMPLE_QUERIES:
    print(f"\nQUERY: {query}")
    result = run_query(query)
    print(f"TOOL:  {result['tool_name']} → {result['tool_result']}")
    print(f"FINAL: {result['final_answer'][:180]}")
    print("-" * 60)

### Exercise 1 — Add a New Tool

Register a `convert_currency` tool on the `MockMCPServer`:

```python
server.register(
    name="convert_currency",
    description="Convert an amount between two currencies. Args: amount (float), from_curr (str), to_curr (str)",
    fn=lambda amount, from_curr, to_curr: f"{amount} {from_curr} = {amount * 1.08:.2f} {to_curr}",
)
```

Then run the query `"Convert 100 USD to EUR"` and verify the agent selects
`convert_currency` automatically — without any other changes to the graph.

**What to observe:** Adding a tool requires zero changes to the agent. This is MCP's
core value proposition — the server catalog evolves independently of the agent code.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────

# TODO: register convert_currency on `server`
# server.register(
#     name="convert_currency",
#     description="TODO: write a clear description with arg names",
#     fn=lambda amount, from_curr, to_curr: f"{amount} {from_curr} = ...",
# )

# TODO: verify the catalog now shows 5 tools
# print([t['name'] for t in server.list_tools()])

# TODO: run a currency query through the full workflow
# result = run_query("Convert 200 GBP to JPY")
# print(result['final_answer'])

---

## Part 6 — Multi-Server Routing

---

Real deployments connect agents to **multiple MCP servers**, each owning a different domain.
The agent aggregates catalogs from all servers, prefixes tool names to avoid collisions,
and routes `call_tool()` to the correct server based on the prefix.

```
                        ┌──────────────────────────┐
                        │   Weather Server          │
                        │   - get_weather           │
  User query            │   - get_forecast          │
      │                 └──────────────────────────┘
      ▼
  ┌──────────┐  routes  ┌──────────────────────────┐
  │  Agent   │ ───────▶ │   Finance Server          │
  └──────────┘          │   - calculate_tip         │
                        │   - convert_currency      │
                        └──────────────────────────┘
```

The aggregated catalog uses `namespace__toolname` keys so the router knows which server
to call without any additional configuration.

In [ ]:
# ─── 6-A: MultiServerClient aggregator ───────────────────────────────────────
# Namespace prefix (e.g. weather__get_weather) prevents tool name collisions across servers

class MultiServerClient:
    """Aggregates tools from multiple MCP servers with namespace prefixing."""

    def __init__(self):
        self._servers: dict[str, MockMCPServer] = {}

    def add_server(self, namespace: str, srv: MockMCPServer):
        """Register a server under a namespace prefix (e.g. 'weather')."""
        self._servers[namespace] = srv

    def list_tools(self) -> list[dict]:
        """Returns all tools across all servers, prefixed with namespace."""
        all_tools = []
        for ns, srv in self._servers.items():
            for tool in srv.list_tools():
                all_tools.append({
                    "name": f"{ns}__{tool['name']}",
                    "description": tool["description"],
                })
        return all_tools

    def call_tool(self, prefixed_name: str, args: dict) -> str:
        """Strips namespace prefix and routes to the correct server."""
        if "__" not in prefixed_name:
            return f"Error: tool name must be namespace__toolname, got '{prefixed_name}'"
        ns, name = prefixed_name.split("__", 1)
        if ns not in self._servers:
            return f"Error: unknown server namespace '{ns}'"
        return self._servers[ns].call_tool(name, args)


# Build two independent servers
weather_server = MockMCPServer()
weather_server.register("get_weather",  "Get weather for a city. Args: city (str)",
                         lambda city: f"Partly cloudy, 18 deg C in {city}")
weather_server.register("get_forecast", "Get 3-day forecast for a city. Args: city (str)",
                         lambda city: f"Mon: 18C, Tue: 20C, Wed: 15C in {city}")

finance_server = MockMCPServer()
finance_server.register("calculate_tip",    "Calculate tip. Args: bill (float), pct (float)",
                          lambda bill, pct: f"${bill * pct / 100:.2f} tip")
finance_server.register("convert_currency", "Convert currency. Args: amount (float), from_curr (str), to_curr (str)",
                          lambda amount, from_curr, to_curr: f"{amount} {from_curr} = {amount * 1.08:.2f} {to_curr}")

# Aggregate into one client
multi_client = MultiServerClient()
multi_client.add_server("weather", weather_server)
multi_client.add_server("finance", finance_server)

print("Aggregated tool catalog:")
for t in multi_client.list_tools():
    print(f"  {t['name']:40} {t['description']}")

In [ ]:
# ─── 6-B: Run the agent against the multi-server client ──────────────────────
# Swap in multi_client as the server — the graph nodes require zero changes.
# This demonstrates that the agent graph is completely server-agnostic.

_original_server = server   # save single-server reference
server = multi_client       # the discover and invoke nodes use the module-level 'server'

multi_queries = [
    "What is the 3-day forecast for Amsterdam?",
    "Convert 500 USD to EUR.",
    "What is the weather in Berlin?",
    "Calculate 18% tip on a $90 dinner.",
]

for q in multi_queries:
    print(f"\nQUERY: {q}")
    result = run_query(q)
    print(f"TOOL:  {result['tool_name']} → {result['tool_result']}")
    print(f"FINAL: {result['final_answer'][:160]}")

server = _original_server   # restore single-server for remaining cells

---

## Part 7 — Error Handling and Fallbacks

---

### Common Failure Modes

| Failure | Cause | Detection | Recovery |
|---------|-------|-----------|----------|
| **Hallucinated tool name** | LLM invents a tool that does not exist | `call_tool` returns error string | Synthesize node surfaces it gracefully |
| **Wrong argument types** | LLM passes string where float expected | `TypeError` caught in `call_tool` | Return descriptive error string |
| **Missing required arg** | LLM omits a required parameter | `TypeError` caught in `call_tool` | Same as above |
| **No matching tool** | Query is out of scope for all registered tools | `tool_name == 'none'` | Synthesize a direct LLM answer |
| **JSON parse failure** | LLM wraps response in markdown fences | `json.JSONDecodeError` | Strip fences and retry |

The key principle: **never let the agent crash on a tool error**. Errors are data — surface
them in the final answer rather than raising exceptions.

In [ ]:
# ─── 7-A: Resilient invoke and synthesize nodes ───────────────────────────────

def invoke_tool_resilient(state: MCPState) -> dict:
    """Node 2 (resilient) — catches all tool errors and surfaces them as strings."""
    if state["tool_name"] == "none":
        print("  [invoke] no tool matched — using direct LLM path")
        return {"tool_result": "__NO_TOOL__"}

    result = server.call_tool(state["tool_name"], state["tool_args"])
    if result.startswith("Error:"):
        print(f"  [invoke] tool error: {result}")
    else:
        print(f"  [invoke] {state['tool_name']} → {result}")
    return {"tool_result": result}


ANSWER_PROMPT_RESILIENT = """User query: {query}
Tool used: {tool_name}
Tool result: {tool_result}

Instructions:
- If tool_result is '__NO_TOOL__', answer the query directly from your own knowledge.
- If tool_result starts with 'Error:', apologise briefly and answer what you can.
- Otherwise, use the tool result to give a concise, helpful answer."""


def synthesize_resilient(state: MCPState) -> dict:
    """Node 3 (resilient) — handles error and no-tool cases."""
    prompt = ANSWER_PROMPT_RESILIENT.format(
        query=state["query"],
        tool_name=state["tool_name"],
        tool_result=state["tool_result"],
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"final_answer": response.content}


# Build the resilient workflow
def create_resilient_workflow():
    graph = StateGraph(MCPState)
    graph.add_node("discover",   discover_and_select)
    graph.add_node("invoke",     invoke_tool_resilient)
    graph.add_node("synthesize", synthesize_resilient)
    graph.set_entry_point("discover")
    graph.add_edge("discover",   "invoke")
    graph.add_edge("invoke",     "synthesize")
    graph.add_edge("synthesize", END)
    return graph.compile()


resilient_app = create_resilient_workflow()

# Test edge cases
edge_cases = [
    "Who won the World Cup in 2022?",    # no matching tool → direct LLM answer
    "What is the weather in Vienna?",    # normal tool path
]

for q in edge_cases:
    print(f"\nQUERY: {q}")
    result = resilient_app.invoke({
        "query": q, "available_tools": [], "tool_name": "",
        "tool_args": {}, "tool_result": "", "final_answer": "",
    })
    print(f"TOOL:  {result['tool_name']} → {result['tool_result']}")
    print(f"FINAL: {result['final_answer'][:200]}")

### Exercise 2 — Argument Validation Recovery

Force the agent to recover from a bad argument error:

1. Register a tool with strict integer args:
   ```python
   server.register("integer_divide", "Divide two integers. Args: a (int), b (int)",
                   lambda a, b: f"{int(a)} // {int(b)} = {int(a) // int(b)}")
   ```
2. Run the query `"Divide 7 by 0"` — division by zero is caught and surfaced as an error string.
3. Observe that `invoke_tool_resilient` catches the error and the agent still responds.
4. **Bonus:** Add a `ValueError` handler in `MockMCPServer.call_tool` for division by zero.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

# TODO: register integer_divide on the server
# server.register(
#     name="integer_divide",
#     description="TODO: write a description with arg names",
#     fn=lambda a, b: f"{int(a)} // {int(b)} = {int(a) // int(b)}",
# )

# TODO: run a query that triggers division by zero
# result = resilient_app.invoke({
#     "query": "Divide 7 by 0",
#     "available_tools": [], "tool_name": "",
#     "tool_args": {}, "tool_result": "", "final_answer": "",
# })
# print(result['final_answer'])

# TODO (bonus): update MockMCPServer.call_tool to also catch ZeroDivisionError
pass

---

## Part 8 ★ — Migration to Real MCP (Bonus)

---

The `MockMCPServer` satisfies the same two-method contract as `langchain-mcp-adapters`.
Swapping to a real server requires **only** changing the server instantiation — the agent
graph, nodes, and state schema stay identical.

### What changes vs the mock

| Aspect | MockMCPServer (this notebook) | Real MCP (langchain-mcp-adapters) |
|--------|------------------------------|-----------------------------------|
| Instantiation | `MockMCPServer()` | `MultiServerMCPClient({...})` async context |
| Tool schema | Description string only | Full JSON Schema per arg |
| Arg selection accuracy | LLM reads free-text description | LLM uses typed JSON Schema (more accurate) |
| Transport | In-process | stdio or HTTP+SSE |
| Agent graph | Linear: discover → invoke → synthesize | Can use `create_react_agent` directly |

The core pattern — discover at runtime, select by name, invoke, synthesize — is identical.

In [ ]:
# ─── 8-A: Real MCP migration sketch ──────────────────────────────────────────
# This cell shows the structural equivalence. It does NOT execute without
# langchain-mcp-adapters installed and a real MCP server running.

REAL_MCP_SKETCH = '''
# Prerequisites:
#   pip install langchain-mcp-adapters
#   node --version  (Node.js required for the filesystem server)

import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

async def main():
    async with MultiServerMCPClient(
        {
            "filesystem": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-filesystem", "/tmp"],
                "transport": "stdio",
            }
        }
    ) as client:
        # get_tools() is the real equivalent of list_tools()
        tools = await client.get_tools()
        print(f"Discovered {len(tools)} tools: {[t.name for t in tools]}")

        # create_react_agent handles discover + invoke + synthesize automatically
        agent = create_react_agent(llm, tools)
        result = await agent.ainvoke({"messages": ["List files in /tmp"]})
        print(result["messages"][-1].content)

asyncio.run(main())
'''

print("Real MCP migration sketch (informational — requires langchain-mcp-adapters):\n")
print(REAL_MCP_SKETCH)

---

## What's Next?

You now understand the MCP client pattern end-to-end. Here's where to go deeper:

### Contrast with hardcoded tools
- **Example 4 — Basic ReAct Agent** (`../4-basic-react-agent/`): Uses `llm.bind_tools([...])` — the
  complement to dynamic MCP discovery. Study both to understand the trade-offs between static
  binding and runtime discovery.

### Scale the pattern
- **Example 6 — Multi-Agent Graph** (`../6-multi-agent-graph/`): Route queries to specialist
  sub-agents. MCP servers can serve as agent boundaries in a multi-agent system.
- **Example 12 — Basic RAG Notebook** (`../12-basic-rag-notebook/`): Combine MCP tool access with
  retrieval — a `search_docs` tool registered on an MCP server is the natural integration point.

### Connect to real servers
- Install `langchain-mcp-adapters` and connect to the filesystem server via `stdio` transport
- Browse the MCP server registry at https://github.com/modelcontextprotocol/servers for
  ready-made servers: GitHub, Slack, Postgres, filesystem, browser control, and more

### Further reading
- Anthropic. (2024). *Model Context Protocol Specification.* https://spec.modelcontextprotocol.io
- Anthropic. (2024). *MCP Introduction.* https://modelcontextprotocol.io/introduction
- LangChain. (2024). *MCP Adapters.* https://github.com/langchain-ai/langchain-mcp-adapters
- MCP Server Registry: https://github.com/modelcontextprotocol/servers

---

*Workshop complete. Next: example 4 (bind_tools contrast) or example 6 (multi-agent architecture).*

---

## Exercise Answer Key

Use this section as a reference **after** attempting the exercises yourself. These are
sample solutions — not the only correct approaches.

### Exercise 1 Answer — Add a New Tool

**Key learning:** Adding a tool requires zero changes to the agent graph, prompts, or state
schema. The only code that changes is the server registration — which is exactly the MCP
separation of concerns in action.

**Writing good descriptions:** The LLM selects tools purely from the description string.
Include the tool's purpose, argument names, and argument types. Vague descriptions cause
the LLM to select the wrong tool or construct bad arguments.

In [ ]:
# Exercise 1 Sample Solution

server.register(
    name="convert_currency",
    description=(
        "Convert an amount between two currencies. "
        "Args: amount (float), from_curr (str, e.g. 'USD'), to_curr (str, e.g. 'EUR')"
    ),
    fn=lambda amount, from_curr, to_curr: (
        f"{amount} {from_curr} = {amount * 1.08:.2f} {to_curr} (approximate)"
    ),
)

print(f"Catalog now has {len(server.list_tools())} tools:")
for t in server.list_tools():
    print(f"  - {t['name']}")

print("\nRunning currency query through the unmodified workflow:")
result = run_query("Convert 200 GBP to JPY")
print(f"Tool:  {result['tool_name']} → {result['tool_result']}")
print(f"Final: {result['final_answer'][:200]}")

# Clean up so remaining cells stay consistent
del server._tools["convert_currency"]

### Exercise 2 Answer — Argument Validation Recovery

**What to observe:** When `MockMCPServer.call_tool` catches an exception, it returns an
`"Error: ..."` string. The resilient synthesize node sees this, acknowledges the failure,
and still produces a useful response instead of crashing.

**The bonus approach:** Catch `ZeroDivisionError` alongside `TypeError` in `call_tool` so
all tool errors produce descriptive strings the synthesize node can reason about.

In [ ]:
# Exercise 2 Sample Solution

# Bonus: extend MockMCPServer.call_tool to also catch ZeroDivisionError
import types

def _call_tool_extended(self, name: str, args: dict) -> str:
    if name not in self._tools:
        return f"Error: unknown tool '{name}'"
    try:
        return str(self._tools[name].fn(**args))
    except ZeroDivisionError:
        return f"Error: division by zero in '{name}'"
    except TypeError as e:
        return f"Error: invalid arguments for '{name}': {e}"

server.call_tool = types.MethodType(_call_tool_extended, server)

server.register(
    name="integer_divide",
    description="Integer-divide two numbers. Args: a (int), b (int)",
    fn=lambda a, b: f"{int(a)} // {int(b)} = {int(a) // int(b)}",
)

print("Testing division by zero recovery:")
result = resilient_app.invoke({
    "query": "Divide 7 by 0",
    "available_tools": [], "tool_name": "",
    "tool_args": {}, "tool_result": "", "final_answer": "",
})
print(f"Tool:  {result['tool_name']} → {result['tool_result']}")
print(f"Final: {result['final_answer'][:200]}")

# Clean up
del server._tools["integer_divide"]